# Deep Convolutional Q-Learning for Pac-Man

## Part 0 - Installing the required packages and importing the libraries

### Installing Gymnasium

In [ ]:
!pip install gymnasium
!pip install "gymnasium[atari, accept-rom-license]"
!pip install ale-py
!apt-get install -y swig
!pip install gymnasium[box2d]

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  swig4.0
Suggested packages:
  swig-doc swig-examples swig4.0-examples swig4.0-doc
The following NEW packages will be installed:
  swig swig4.0
0 upgraded, 2 newly installed, 0 to remove and 35 not upgraded.
Need to get 1,116 kB of archives.
After this operation, 5,542 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig4.0 amd64 4.0.2-1ubuntu1 [1,110 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 swig all 4.0.2-1ubuntu1 [5,632 B]
Fetched 1,116 kB in 2s (477 kB/s)
Selecting previously unselected package swig4.0.
(Reading database ... 126371 files and directories currently installed.)
Preparing to unpack .../swig4.0_4.0.2-1ubuntu1_amd64.deb ...
Unpacking swig4.0 (4.0.2-1ubuntu1) ...
Selecting previously unselected package swig.
Preparing to unpack .../swig_4.0.2-1ubunt

### Importing the libraries

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
from torch.utils.data import DataLoader, TensorDataset

## Part 1 - Building the AI

### Creating the architecture of the Neural Network

In [ ]:
# define a convolutional neural network
class Network(nn.Module):
    def __init__(self, action_size, seed=42):
      super(Network, self).__init__()

      # set random seed for reproducibility
      self.seed = torch.manual_seed(seed)

      # --- convolutional feature extractor ---
      # input shape is assumed to be (3, H, W), e.g. RGB image

      # 1st conv layer: 3 input channels -> 32 feature maps
      # kernel size = 8, stride = 4 (downsamples heavily)
      self.conv1 = nn.Conv2d(3, 32, kernel_size=8, stride=4)
      self.bn1 = nn.BatchNorm2d(32)  # normalise activations for stability

      # 2nd conv layer: 32 -> 64 channels
      # kernel size = 4, stride = 2 (further downsampling)
      self.conv2 = nn.Conv2d(32, 64, kernel_size=4, stride=2)
      self.bn2 = nn.BatchNorm2d(64)

      # 3rd conv layer: 64 -> 64 channels
      # kernel size = 3, stride = 1 (keeps spatial resolution)
      self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=1)
      self.bn3 = nn.BatchNorm2d(64)

      # 4th conv layer: 64 -> 128 channels
      # kernel size = 3, stride = 1
      self.conv4 = nn.Conv2d(64, 128, kernel_size=3, stride=1)
      self.bn4 = nn.BatchNorm2d(128)

      # --- fully connected layers (classifier / Q-value head) ---
      # flattened size after conv layers is assumed to be 128*10*10
      self.fc1 = nn.Linear(128 * 10 * 10, 512)
      self.fc2 = nn.Linear(512, 256)

      # output layer: number of possible actions (Q-values or logits)
      self.fc3 = nn.Linear(256, action_size)

    def forward(self, state):
        # pass input through convolutional layers with ReLU activation
        x = F.relu(self.bn1(self.conv1(state)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))

        # flatten from (batch, channels, H, W) -> (batch, features)
        x = x.view(x.size(0), -1)

        # pass through fully connected layers
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        # final layer outputs action values (no activation here)
        return self.fc3(x)

## Part 2 - Training the AI

### Setting up the environment

In [ ]:
import ale_py
import gymnasium as gym

# create the ms pacman environment
# deterministic version makes the game behave the same each time
# full_action_space = False means we use a smaller set of actions
env = gym.make('MsPacmanNoFrameskip-v0', full_action_space = False)

# get the shape of the state (the screen image)
state_shape = env.observation_space.shape

# get the size of the state (height of the image in pixels, since we only take the first element)
state_size = env.observation_space.shape[0]

# get how many actions the agent can take in this game
number_actions = env.action_space.n

# print out basic information about the environment
print('State shape: ', state_shape)
print('State size: ', state_size)
print('Number of actions: ', number_actions)


State shape:  (210, 160, 3)
State size:  210
Number of actions:  9


/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment MsPacmanNoFrameskip-v0 is out of date. You should consider upgrading to version `v4`.
  logger.deprecation(


### Initializing the hyperparameters

In [ ]:
learning_rate = 5e-4
minibatch_size = 64
discount_factor = 0.99

### Preprocessing the frames

In [ ]:
from PIL import Image
from torchvision import transforms

def preprocess_frame(frame):
    # convert the numpy array (game frame) into a PIL image
    frame = Image.fromarray(frame)

    # define a sequence of image transformations
    preprocess = transforms.Compose([
        transforms.Resize((128, 128)),  # resize the image to 128x128 pixels
        transforms.ToTensor()           # convert the image to a torch tensor (values in [0,1])
    ])

    # apply the transformations and add a batch dimension (so shape is [1, channels, height, width])
    return preprocess(frame).unsqueeze(0)

### Implementing the DCQN class

In [ ]:
class Agent():

    def __init__(self, action_size):
        # use gpu if available, otherwise fall back to cpu
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        # number of actions available in the environment
        self.action_size = action_size

        # two q-networks: local (for acting) and target (for stable learning)
        self.local_qnetwork = Network(action_size).to(self.device)
        self.target_qnetwork = Network(action_size).to(self.device)

        # optimiser for training the local network
        self.optimizer = optim.Adam(self.local_qnetwork.parameters(), lr=learning_rate)

        # replay buffer to store past experiences
        self.memory = deque(maxlen=10000)

    def step(self, state, action, reward, next_state, done):
        # preprocess the raw frames into tensors
        state = preprocess_frame(state)
        next_state = preprocess_frame(next_state)

        # save the experience to memory
        self.memory.append((state, action, reward, next_state, done))

        # once enough samples are collected, start training
        if len(self.memory) > minibatch_size:
            experiences = random.sample(self.memory, k=minibatch_size)
            self.learn(experiences, discount_factor)

    def act(self, state, epsilon=0.):
        # preprocess state and move it to gpu/cpu
        state = preprocess_frame(state).to(self.device)

        # switch to evaluation mode to pick an action without gradients
        self.local_qnetwork.eval()
        with torch.no_grad():
            action_values = self.local_qnetwork(state)
        self.local_qnetwork.train()

        # epsilon-greedy strategy: mostly exploit, sometimes explore
        if random.random() > epsilon:
            return np.argmax(action_values.cpu().data.numpy())  # pick best action
        else:
            return random.choice(np.arange(self.action_size))  # pick random action

    def learn(self, experiences, discount_factor):
        # unpack batch of experiences into separate arrays
        states, actions, rewards, next_states, dones = zip(*experiences)

        # stack and convert to torch tensors on the right device
        states = torch.from_numpy(np.vstack(states)).float().to(self.device)
        actions = torch.from_numpy(np.vstack(actions)).long().to(self.device)
        rewards = torch.from_numpy(np.vstack(rewards)).float().to(self.device)
        next_states = torch.from_numpy(np.vstack(next_states)).float().to(self.device)
        dones = torch.from_numpy(np.vstack(dones).astype(np.uint8)).float().to(self.device)

        # get maximum predicted q values for next states from the target network
        next_q_targets = self.target_qnetwork(next_states).detach().max(1)[0].unsqueeze(1)

        # compute q targets (reward + discounted future return if not done)
        q_targets = rewards + discount_factor * next_q_targets * (1 - dones)

        # get expected q values for chosen actions from the local network
        q_expected = self.local_qnetwork(states).gather(1, actions)

        # calculate mean squared error loss
        loss = F.mse_loss(q_expected, q_targets)

        # backpropagation step to update the local network
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

### Initializing the DCQN agent

In [ ]:
agent = Agent(number_actions)

### Training the DCQN agent

In [9]:
# training settings
number_episodes = 1000 # total number of episodes to run
maximum_number_timesteps_per_episode = 10000 # limit steps per episode
epsilon_starting_value  = 1.0 # starting epsilon (fully random actions at the start)
epsilon_ending_value  = 0.01 # minimum epsilon (almost always greedy later)
epsilon_decay_value  = 0.995 # how quickly epsilon decreases
epsilon = epsilon_starting_value
scores_on_100_episodes = deque(maxlen=100) # store last 100 scores for averaging

for episode in range(1, number_episodes + 1):
    # reset the environment for a new episode
    state, _ = env.reset()
    score = 0

    # run steps until episode finishes or max timesteps reached
    for t in range(maximum_number_timesteps_per_episode):
        # choose an action using epsilon-greedy
        action = agent.act(state, epsilon)

        # take the action in the environment
        next_state, reward, done, _, _ = env.step(action)

        # save experience and possibly learn from a batch
        agent.step(state, action, reward, next_state, done)

        # move to the next state
        state = next_state

        # update total reward for this episode
        score += reward

        # stop if episode has finished
        if done:
            break

    # keep track of scores across the last 100 episodes
    scores_on_100_episodes.append(score)

    # reduce epsilon (but not below the minimum)
    epsilon = max(epsilon_ending_value, epsilon_decay_value * epsilon)

    # print progress on the same line
    print('\rEpisode {}\tAverage Score: {:.2f}'.format(episode, np.mean(scores_on_100_episodes)), end="")

    # every 100 episodes, print progress on a new line
    if episode % 100 == 0:
        print('\rEpisode {}\tAverage Score: {:.2f}'.format(episode, np.mean(scores_on_100_episodes)))

    # if the environment is solved (average score ≥ 500 over last 100 episodes), stop training
    if np.mean(scores_on_100_episodes) >= 500.0:
        print('\nEnvironment solved in {:d} episodes!\tAverage Score: {:.2f}'.format(episode - 100, np.mean(scores_on_100_episodes)))
        torch.save(agent.local_qnetwork.state_dict(), 'checkpoint.pth')  # save trained model
        break

Episode 100	Average Score: 216.20
Episode 200	Average Score: 334.40
Episode 300	Average Score: 480.00
Episode 400	Average Score: 350.10
Episode 500	Average Score: 460.40
Episode 554	Average Score: 511.00
Environment solved in 454 episodes!	Average Score: 511.00


## Part 3 - Visualizing the results

In [14]:
import glob
import io
import base64
import imageio
from IPython.display import HTML, display

def show_video_of_model(agent, env_name):
    # create the env in rgb_array mode so render() returns frames as numpy arrays
    env = gym.make(env_name, render_mode='rgb_array')

    # reset the env to get the initial observation
    state, _ = env.reset()

    # keep stepping until the episode ends
    done = False
    frames = []

    while not done:
        # grab the current frame for the video
        frame = env.render()
        frames.append(frame)

        # choose an action from the agent (default is greedy: epsilon=0.)
        action = agent.act(state)

        # step the env with the chosen action
        # note: gymnasium returns (obs, reward, terminated, truncated, info)
        state, reward, done, _, _ = env.step(action)

    # tidy up the env
    env.close()

    # save all collected frames as an mp4 video
    imageio.mimsave('video.mp4', frames, fps=30)

# run one episode and record the video
show_video_of_model(agent, 'MsPacmanNoFrameskip-v0')

def show_video():
    # look for any mp4 files in the current folder
    mp4list = glob.glob('*.mp4')

    if len(mp4list) > 0:
        # take the first mp4 we find
        mp4 = mp4list[0]

        # read the file and base64-encode it for inline display
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)

        # display the video in the notebook cell
        display(HTML(data='''
            <video alt="test" autoplay loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
            </video>'''.format(encoded.decode('ascii'))))
    else:
        # if no mp4 is found, tell the user
        print("could not find video")

# display the most recent (or first found) mp4 inline
show_video()
""

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment MsPacmanNoFrameskip-v0 is out of date. You should consider upgrading to version `v4`.
  logger.deprecation(


''